# Adversarial Attractors Analysis

This notebook investigates the phenomenon of "Adversarial Attractors" in Convolutional Neural Networks. 

When performing untargeted attacks (like FGSM) on a diverse dataset, the resulting adversarial classes are rarely uniformly distributed. Instead, the gradient descent often pushes the perturbed images towards a small subset of specific classes. This happens due to the inherent topology of the model's latent space, where certain regions (classes) have wider, more accessible decision boundaries.

We will attack 100 random images from the MiniImageNet dataset and analyze the distribution of the resulting adversarial classifications across three different architectures.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from collections import Counter

# Import models
from tensorflow.keras.applications import EfficientNetB0, InceptionV3, MobileNetV2

# Import preprocessing and decoding
from tensorflow.keras.applications.efficientnet import preprocess_input as preprocess_eff, decode_predictions as decode_eff
from tensorflow.keras.applications.inception_v3 import preprocess_input as preprocess_inc, decode_predictions as decode_inc
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mob, decode_predictions as decode_mob

In [ ]:
# Dictionary with model configurations
models_dict = {
    'MobileNetV2': {
        'model': MobileNetV2(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_mob,
        'decode_fn': decode_mob,
        'eps_scale': 1.0
    },
    'EfficientNetB0': {
        'model': EfficientNetB0(weights='imagenet'),
        'target_size': (224, 224),
        'preprocess_fn': preprocess_eff,
        'decode_fn': decode_eff,
        'eps_scale': 127.5
    },
    'InceptionV3': {
        'model': InceptionV3(weights='imagenet'),
        'target_size': (299, 299),
        'preprocess_fn': preprocess_inc,
        'decode_fn': decode_inc,
        'eps_scale': 1.0
    }
}

# Freeze weights as we only need to perform inference and compute gradients w.r.t inputs
for config in models_dict.values():
    config['model'].trainable = False

### Helper Functions & Dataset Loading
We load 100 random images (one from each MiniImageNet class).

In [ ]:
def load_and_preprocess_image(img_path, target_size, preprocess_fn):
    img_raw = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_raw, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, target_size)
    img = preprocess_fn(img) # Dynamic preprocessing
    img = tf.expand_dims(img, axis=0) # Add batch dimension
    return img

def get_imagenet_label(prob, decode_fn):
    return decode_fn(prob, top=1)[0][0][1] # Return just the string name

In [ ]:
# Define the path to the 100 images folder
dataset_path = '../miniimagenet_random_100/'
image_paths = glob.glob(os.path.join(dataset_path, '*.*'))

# Limit to 100 just in case
image_paths = image_paths[:100]
print(f"Total images loaded for the experiment: {len(image_paths)}")

In [ ]:
loss_object = tf.keras.losses.CategoricalCrossentropy()

def fgsm_attack(input_image, input_label, epsilon, current_model):
    with tf.GradientTape() as tape:
        tape.watch(input_image)
        prediction = current_model(input_image)
        loss = loss_object(input_label, prediction)
    
    # Get gradients of the loss w.r.t to the input image
    gradient = tape.gradient(loss, input_image)
    
    # Get the sign of the gradients
    signed_grad = tf.sign(gradient)
    
    # Create the adversarial image
    adv_image = input_image + epsilon * signed_grad
    return adv_image

### Execute FGSM Attack on the Dataset
We iterate through the dataset for each model. We record the original prediction and the adversarial prediction. We only count the attack as an "attractor" mapping if the attack was successful (i.e., the class actually changed).

In [ ]:
base_epsilon = 0.01 # Magnitude of the attack

# Dictionary to store the results
attractor_results = {}

for model_name, m_config in models_dict.items():
    # Print a clear header when switching models
    print(f"\n{'='*80}")
    print(f"ANALYZING ADVERSARIAL ATTRACTORS FOR: {model_name}")
    print(f"{'='*80}")
    
    current_model = m_config['model']
    target_size = m_config['target_size']
    preprocess_fn = m_config['preprocess_fn']
    decode_fn = m_config['decode_fn']
    epsilon = base_epsilon * m_config['eps_scale']
    
    successful_adv_classes = []
    
    for i, img_path in enumerate(image_paths):
        # Progress tracker
        if (i+1) % 20 == 0:
            print(f"Processing image {i+1}/{len(image_paths)}...")
            
        input_img = load_and_preprocess_image(img_path, target_size, preprocess_fn)
        
        # Original Prediction
        original_probs = current_model(input_img, training=False)
        orig_class_idx = tf.argmax(original_probs, axis=-1).numpy()[0]
        orig_class_name = get_imagenet_label(original_probs.numpy(), decode_fn)
        
        # Create One-Hot Label
        input_label = tf.reshape(tf.one_hot(orig_class_idx, original_probs.shape[-1]), (1, -1))
        
        # FGSM Attack
        adv_img = fgsm_attack(input_img, input_label, epsilon, current_model)
        
        # Adversarial Prediction
        adv_probs = current_model(adv_img, training=False)
        adv_class_idx = tf.argmax(adv_probs, axis=-1).numpy()[0]
        
        # Only record if the attack was successful
        if orig_class_idx != adv_class_idx:
            adv_class_name = get_imagenet_label(adv_probs.numpy(), decode_fn)
            successful_adv_classes.append(adv_class_name)
            
    # Count frequencies of the sink classes
    class_counts = Counter(successful_adv_classes)
    attractor_results[model_name] = {
        'counts': class_counts,
        'total_success': len(successful_adv_classes)
    }
    
    print(f"-> Successful attacks: {len(successful_adv_classes)}/{len(image_paths)}")
    if len(successful_adv_classes) > 0:
        top_1 = class_counts.most_common(1)[0]
        print(f"-> Top Attractor: '{top_1[0]}' ({top_1[1]} times)")